# Alpha Go Everywhere — NN1–NN5 Full Pipeline (Optimised)

Same model architecture, loss, L1 penalty, early stopping, and ensemble logic as the paper.  
All speed gains come from data handling — no ML changes.

### What changed vs. the original notebook
- Pre-convert all data to tensors **once** at load time (no per-cycle DataFrame copies)
- Manual batch slicing instead of DataLoader (zero overhead for in-memory data)
- Pre-cache L1 weight references (skip `model.modules()` iteration every step)
- `BATCH_SIZE = 50,000` (5× fewer gradient steps per epoch)
- `RETRAIN_EVERY = 3` (12 cycles instead of 35)

### What is identical
- NN1–NN5 architectures (Linear → BatchNorm → ReLU)
- Xavier init, Adam lr=0.001, L1 λ=1e-4 on Linear weights only
- Early stopping patience=5, max 100 epochs
- Ensemble of 5, expanding window, 10-year validation

In [14]:
# ═══════════════════════════════════════════════════════════════
# 1 — Imports & Device
# ═══════════════════════════════════════════════════════════════
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import time
from pathlib import Path
import platform

print("machine:", platform.machine())   # should ideally say arm64
print("mps available:", torch.backends.mps.is_available())

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("Using device:", DEVICE)

machine: arm64
mps available: True
Using device: mps


In [15]:
# ═══════════════════════════════════════════════════════════════
# 2 — Configuration
# ═══════════════════════════════════════════════════════════════
PARQUET_PATH    = Path(r'/Users/joshaijer/Documents/factor_model_project/normalized/USA_ranked.parquet')
OUTPUT_DIR      = Path('.')
OUTPUT_DIR.mkdir(exist_ok=True)

START_TEST_YEAR = 1990
END_TEST_YEAR   = 1993
RETRAIN_EVERY   = 1

ENSEMBLE_SIZE   = 5
BATCH_SIZE      = 10_000
MAX_EPOCHS      = 100
PATIENCE        = 5
LR              = 0.01
L1_CANDIDATES   = [1e-5, 1e-4, 1e-3]

MODELS_TO_RUN   = ['NN1', 'NN2', 'NN3', 'NN4', 'NN5']

ARCHITECTURES = {
    'NN1': [32],
    'NN2': [32, 16],
    'NN3': [32, 16, 8],
    'NN4': [32, 16, 8, 4],
    'NN5': [32, 16, 8, 4, 2],
}
print('Config loaded.')

Config loaded.


In [16]:
# ═══════════════════════════════════════════════════════════════
# 3 — Load data & convert to tensors ONCE
# ═══════════════════════════════════════════════════════════════
raw = pd.read_parquet(PARQUET_PATH)
raw.columns = [c.strip().upper() for c in raw.columns]
raw['DATE'] = pd.to_datetime(raw['DATE'])

NON_FEAT = {'PERMNO', 'DATE', 'TARGET', 'YEAR'}
FEAT_COLS = [c for c in raw.columns if c not in NON_FEAT]
INPUT_DIM = len(FEAT_COLS)

# Pre-compute year array (int32) — used for fast splitting
YEARS = raw['DATE'].dt.year.values.astype(np.int32)

# Convert features + target to contiguous float32 tensors once
X_ALL = torch.tensor(
    raw[FEAT_COLS].fillna(0).replace([np.inf, -np.inf], 0).values,
    dtype=torch.float32
).contiguous()

Y_ALL = torch.tensor(
    raw['TARGET'].values, dtype=torch.float32
).contiguous()

# Keep PERMNO and DATE as numpy for result building
PERMNO_ALL = raw['PERMNO'].values
DATE_ALL   = raw['DATE'].values

del raw  # free ~1GB

print(f'Rows: {len(X_ALL):,}  |  Features: {INPUT_DIM}  |  '
      f'Years: {YEARS.min()}–{YEARS.max()}')

/var/folders/1z/df3ckyd548j1xwlq0x0vds080000gn/T/ipykernel_75620/965903454.py:6: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  raw['DATE'] = pd.to_datetime(raw['DATE'])


Rows: 3,433,378  |  Features: 36  |  Years: 1964–2024


In [17]:
# ═══════════════════════════════════════════════════════════════
# 4 — Model definition (identical to paper)
# ═══════════════════════════════════════════════════════════════
class FlexibleNN(nn.Module):
    def __init__(self, input_size, hidden_sizes):
        super().__init__()
        layers = []
        prev = input_size
        for h in hidden_sizes:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU()]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

def weights_init(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight)
        nn.init.constant_(m.bias, 0.0)

for name, sizes in ARCHITECTURES.items():
    n = sum(p.numel() for p in FlexibleNN(INPUT_DIM, sizes).parameters())
    print(f'{name}  {sizes}  params={n:,}')

NN1  [32]  params=1,281
NN2  [32, 16]  params=1,825
NN3  [32, 16, 8]  params=1,969
NN4  [32, 16, 8, 4]  params=2,009
NN5  [32, 16, 8, 4, 2]  params=2,021


In [18]:
# ═══════════════════════════════════════════════════════════════
# 5 — Training engine (no DataLoader, manual batching)
# ═══════════════════════════════════════════════════════════════
def train_single(model, X_tr, y_tr, X_va, y_va, lambda1):
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.MSELoss()
    n_tr = len(X_tr)
    h = n_tr // BATCH_SIZE + 1
    best_val = float('inf')
    patience_ctr = 0
    best_state = None
    stopped = MAX_EPOCHS

    for epoch in range(MAX_EPOCHS):
        model.train()
        h = n_tr // BATCH_SIZE + 1
        perm = torch.randperm(n_tr)
        for _ in range(h):
            i = torch.randint(0, max(1, n_tr - BATCH_SIZE), (1,)).item()
            idx = perm[i:i + BATCH_SIZE]
            xb, yb = X_tr[idx], y_tr[idx]
            optimizer.zero_grad()
            l1_reg = sum(p.abs().sum() for p in model.parameters())
            loss = criterion(model(xb), yb) + lambda1 * l1_reg
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_va), y_va).item()

        if val_loss < best_val:
            best_val = val_loss
            patience_ctr = 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1
        if patience_ctr >= PATIENCE:
            stopped = epoch + 1
            break

    if best_state:
        model.load_state_dict(best_state)
    return model, stopped, best_val


def train_ensemble(name, X_tr, y_tr, X_va, y_va, test_lo):
    sizes = ARCHITECTURES[name]
    models, epochs = [], []
    for seed in range(ENSEMBLE_SIZE):
        torch.manual_seed(seed)
        best_val = float('inf')
        best_model = None
        best_ep = 0
        # Search over L1 candidates — pick best per seed
        for lam in L1_CANDIDATES:
            m = FlexibleNN(INPUT_DIM, sizes)
            m.apply(weights_init)
            m, ep, val = train_single(m, X_tr, y_tr, X_va, y_va, lam)
            if val < best_val:
                best_val = val
                best_model = m
                best_ep = ep
        models.append(best_model)
        epochs.append(best_ep)
        # Save weights for international market prediction (Table 1)
        save_dir = OUTPUT_DIR / 'model_weights' / f'{name}'
        save_dir.mkdir(parents=True, exist_ok=True)
        for i, m in enumerate(models):
            torch.save(m.state_dict(), save_dir / f'year{test_lo}_seed{i}.pt')
    return models, epochs


@torch.inference_mode()
def predict_ensemble(ensemble, X_te):
    """Average predictions. Single forward pass per model (no batching)."""
    preds = torch.zeros(len(X_te))
    for m in ensemble:
        m.eval()
        preds += m(X_te).squeeze()
    return (preds / len(ensemble)).numpy()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 6 — Main loop
# ═══════════════════════════════════════════════════════════════
forecasts  = {n: [] for n in MODELS_TO_RUN}
timing_log = []
total_start = time.time()

first_train_end = START_TEST_YEAR - 11  # 1979

for train_end in range(first_train_end, END_TEST_YEAR, RETRAIN_EVERY):
    val_end   = train_end + 10
    test_lo   = val_end + 1
    test_hi   = val_end + RETRAIN_EVERY

    # ── Fast integer mask splitting (no pandas) ──
    tr_mask = YEARS <= train_end
    va_mask = (YEARS > train_end) & (YEARS <= val_end)
    te_mask = (YEARS > val_end) & (YEARS <= test_hi)

    n_tr, n_va, n_te = tr_mask.sum(), va_mask.sum(), te_mask.sum()
    if n_tr == 0 or n_va == 0 or n_te == 0:
        print(f'⏭  Skip {test_lo}–{test_hi}  '
              f'(tr={n_tr}, va={n_va}, te={n_te})')
        continue

    X_tr, y_tr = X_ALL[tr_mask], Y_ALL[tr_mask].unsqueeze(1)
    X_va, y_va = X_ALL[va_mask], Y_ALL[va_mask].unsqueeze(1)
    X_te       = X_ALL[te_mask]

    # Test metadata for result building
    te_permno = PERMNO_ALL[te_mask]
    te_date   = DATE_ALL[te_mask]
    te_target = Y_ALL[te_mask].numpy()

    print(f"\n{'═'*72}")
    print(f'  TEST {test_lo}–{test_hi}  │  '
          f'tr={n_tr:,}  va={n_va:,}  te={n_te:,}  │  '
          f'batches/epoch={n_tr//BATCH_SIZE + 1}')
    print(f"{'═'*72}")

    cycle_start = time.time()

    for idx, name in enumerate(MODELS_TO_RUN):
        t0 = time.time()

        ensemble, epochs = train_ensemble(name, X_tr, y_tr, X_va, y_va, test_lo)
        preds = predict_ensemble(ensemble, X_te)

        # Store results
        forecasts[name].append(pd.DataFrame({
            'PERMNO': te_permno,
            'DATE': te_date,
            'TARGET': te_target,
            'prediction': preds,
        }))

        elapsed = time.time() - t0
        mean_ep = np.mean(epochs)
        timing_log.append((f'{test_lo}-{test_hi}', name, elapsed, epochs))

        # ── Progress ──
        cycle_elapsed = time.time() - cycle_start
        cycle_eta = cycle_elapsed / (idx+1) * (len(MODELS_TO_RUN)-idx-1)
        print(f'  {name}  │ {elapsed:5.0f}s  │ ep:{mean_ep:4.1f}  │ '
              f'pred:[{preds.min():.4f},{preds.max():.4f}]  │ '
              f'cycle ETA:{cycle_eta/60:.1f}m')

    # ── Cycle summary + incremental save ──
    cycle_t = time.time() - cycle_start
    total_t = time.time() - total_start
    done = (train_end - first_train_end) // RETRAIN_EVERY + 1
    total_cycles = len(range(first_train_end, END_TEST_YEAR, RETRAIN_EVERY))
    eta_hr = (total_t / done) * (total_cycles - done) / 3600

    print(f'  ── {cycle_t/60:.1f}min  │  {done}/{total_cycles} cycles  │  '
          f'ETA: {eta_hr:.1f}hr ──')

    for name in MODELS_TO_RUN:
        pd.concat(forecasts[name]).to_csv(
            OUTPUT_DIR / f'USA_{name}_forecasts.csv', index=False)

print(f"\n{'═'*72}")
print(f'  DONE — {(time.time()-total_start)/60:.1f} minutes')
print(f"{'═'*72}")


════════════════════════════════════════════════════════════════════════
  TEST 1990–1990  │  tr=596,830  va=630,079  te=65,070  │  batches/epoch=60
════════════════════════════════════════════════════════════════════════
  NN1  │    35s  │ ep:15.2  │ pred:[-0.0255,0.0423]  │ cycle ETA:2.3m
  NN2  │    40s  │ ep:16.4  │ pred:[-0.0693,0.0608]  │ cycle ETA:1.9m
  NN3  │    47s  │ ep:15.6  │ pred:[-0.0316,0.0415]  │ cycle ETA:1.4m
  NN4  │    63s  │ ep:20.2  │ pred:[-0.0484,0.0509]  │ cycle ETA:0.8m
  NN5  │    70s  │ ep:12.6  │ pred:[-0.0175,0.0295]  │ cycle ETA:0.0m
  ── 4.2min  │  1/14 cycles  │  ETA: 0.9hr ──

════════════════════════════════════════════════════════════════════════
  TEST 1991–1991  │  tr=648,533  va=643,446  te=64,330  │  batches/epoch=65
════════════════════════════════════════════════════════════════════════
  NN1  │    36s  │ ep:22.4  │ pred:[-0.0358,0.0459]  │ cycle ETA:2.4m
  NN2  │    51s  │ ep:18.6  │ pred:[-0.0356,0.0403]  │ cycle ETA:2.2m
  NN3  │    58s  │

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 7 — Save & Summary
# ═══════════════════════════════════════════════════════════════
print('Final files:\n')
for name in MODELS_TO_RUN:
    df = pd.concat(forecasts[name])
    p = OUTPUT_DIR / f'USA_{name}_forecasts.csv'
    df.to_csv(p, index=False)
    print(f'  {p.name:30s}  │  {len(df):>10,} rows  │  {df["DATE"].nunique():>4} months')

print('\nPerformance:\n')
for name in MODELS_TO_RUN:
    df = pd.concat(forecasts[name])
    yt, yp = df['TARGET'].values, df['prediction'].values
    sse = np.sum((yt - yp)**2)
    sst = np.sum(yt**2)
    r2 = 1 - sse/sst if sst > 0 else np.nan
    corr = np.corrcoef(yt, yp)[0,1]
    print(f'  {name}  │  R²oos={r2*100:.3f}%  │  corr={corr:.4f}  │  '
          f'pred μ={yp.mean():.5f}  σ={yp.std():.5f}')

Final files:

  USA_NN1_forecasts.csv           │   2,206,469 rows  │   419 months
  USA_NN2_forecasts.csv           │   2,206,469 rows  │   419 months
  USA_NN3_forecasts.csv           │   2,206,469 rows  │   419 months
  USA_NN4_forecasts.csv           │   2,206,469 rows  │   419 months
  USA_NN5_forecasts.csv           │   2,206,469 rows  │   419 months

Performance:

  NN1  │  R²oos=0.605%  │  corr=0.0802  │  pred μ=0.00288  σ=0.01257
  NN2  │  R²oos=0.543%  │  corr=0.0752  │  pred μ=0.00216  σ=0.01224
  NN3  │  R²oos=0.618%  │  corr=0.0808  │  pred μ=0.00271  σ=0.01244
  NN4  │  R²oos=0.580%  │  corr=0.0787  │  pred μ=0.00292  σ=0.01185
  NN5  │  R²oos=0.520%  │  corr=0.0759  │  pred μ=0.00337  σ=0.01086


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 8 — Timing
# ═══════════════════════════════════════════════════════════════
tdf = pd.DataFrame(timing_log, columns=['Cycle','Model','Seconds','Epochs'])
tdf['Avg_Ep'] = tdf['Epochs'].apply(np.mean)
print(tdf.groupby('Model').agg(
    Total_s=('Seconds','sum'), Mean_s=('Seconds','mean'),
    Mean_Ep=('Avg_Ep','mean')
).round(1))

       Total_s  Mean_s  Mean_Ep
Model                          
NN1     1492.9    42.7     17.4
NN2     2158.7    61.7     17.0
NN3     2726.5    77.9     17.3
NN4     3209.3    91.7     16.4
NN5     3595.5   102.7     13.9


/var/folders/1z/df3ckyd548j1xwlq0x0vds080000gn/T/ipykernel_41694/3203270893.py:5: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  tdf['Avg_Ep'] = tdf['Epochs'].apply(np.mean)
